In [1]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities import extractor
import uproot
import awkward as ak    

x_MH250=extractor("Dati/Tprime_tAq_1800_MH250_LH_2017.root", "Events")


file=uproot.open("Dati/Tprime_tAq_1800_MH250_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]

#Filtriamo i dati

mask = ak.flatten(Fatjet_isMatchedWithA) == 1
x_filtered = x_MH250[mask]


/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/riccardo/anaconda3/envs/rootnev/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "
/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


In [5]:
from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()

def voigt(x, norm, mu, sigma, gamma):
    return voigt_profile(x-mu, sigma, gamma) * norm

bin_counts, bin_edges = np.histogram(x_plot, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_plot) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_plot) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt)

m_voigt=Minuit(ls_voigt,  norm=1, mu=250, sigma=5, gamma=1)
m_voigt.limits["mu"]= (225, 275)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.migrad()


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 3.517e+04 (χ²/ndof = 764.6)│              Nfcn = 483              │
│ EDM = 0.000117 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm  │  0.3523   │  0.0026   │            │            │         │         │       │
│ 1 │ mu    │  237.99   │   0.12    │            │            │   225   │   275   │       │
│ 2 │ sigma │   6.12    │   0.05    │            │            │   0.1   │   20    │       │
│ 3 │ gamma │  10.0e-3  │  0.9e-3   │            │            │  0.01   │   10    │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬─────────────────────────────────────┐
│       │     norm       mu    sigma    gamma │
├───────┼─────────────────────────────────────┤
│  norm │ 6.49e-06     1e-6    -0e-6 -0.01e-9 │
│    mu │     1e-6   0.0131  -0.0056  0.30e-9 │
│ sigma │    -0e-6  -0.0056   0.0028 -0.86e-9 │
│ gamma │ -0.01e-9  0.30e-9 -0.86e-9 4.87e-11 │
└───────┴─────────────────────────────────────┘

In [3]:
fit_MH250_values={}
fit_MH250_errors={}

fit_values={'MH250': fit_MH250_values,}
fit_errors={'MH250_errors': fit_MH250_errors}

for param in m_voigt.parameters:
    fit_MH250_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deveestarre gli errori 
    fit_MH250_errors[error] = m_voigt.errors[error]

print(fit_MH250_values)
print(fit_MH250_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH250"]=fit_MH250_values
results["MH250_errors"]=fit_MH250_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH250"]=fit_MH250_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH250_errors"]=fit_MH250_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  


{'norm': 0.35232240742169235, 'mu': 237.98921271161743, 'sigma': 6.118577260002192, 'gamma': 0.010000013713667622}
{'norm': 0.0025476002182494364, 'mu': 0.11465250491973222, 'sigma': 0.05291710535381844, 'gamma': 0.000887008902188167}
